In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai

### Read amazon sample dataset 

In [ ]:
df_items = pd.read_json('../../data/meta_Electronics_2022_onwards_with_ratings_100_sample_1000.jsonl', lines=True)
df_items.head()

In [ ]:
from typing import Hashable, Any
list[tuple[Hashable, Any]](df_items["features"].items())[0]

In [ ]:
list(df_items["images"].items())[0]

### Sample 50 items from sample dataset

In [ ]:
# concate title and features as description
def preprocess_description(row):
    return f"{row['title']} {row['features']}"

# extract larges image from images list
def extract_largest_image(row):
    return row["images"][0].get("large", "")


In [ ]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_largest_image, axis=1)

In [ ]:
df_items.head()

In [ ]:
df_sample_50 = df_items.sample(50, random_state=20)

In [ ]:
len(df_sample_50)

In [ ]:
data_to_embed = df_sample_50[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

### define embedding function

In [ ]:
response = openai.embeddings.create(
    model="text-embedding-3-small",
    input=data_to_embed[0]["description"]
)
response.data[0].embedding


In [ ]:
def create_embeddings(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding


### Create Local Qdrant Vector database

In [ ]:
qdrant_client = QdrantClient(
    url="http://localhost:6333",
)


In [ ]:
qdrant_client.create_collection(
    collection_name="amazon_items-collection-00",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)



In [ ]:
# embed data
point_struct = PointStruct(
    id=0,
    vector=create_embeddings("I am kitta"),
    payload={
        "text": "I am kitta",
        "model": "text-embedding-3-small"
    },
)

point_struct




In [ ]:
point_structs = []
for i, data in enumerate(data_to_embed):
    embedding = create_embeddings(data["description"])
    point_structs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )
    

In [ ]:
len(data_to_embed)

### insert embeddings to vector database (Qdrant)

In [ ]:
qdrant_client.upsert(
    collection_name="amazon_items-collection-00",
    points=point_structs,
    wait=True
)




In [ ]:
# retrieve embeddings from vector database
def retrieve_embeddings(query, collection_name="amazon_items-collection-00", k=5):
    response = qdrant_client.query_points(
        collection_name=collection_name,
        query=create_embeddings(query),
        limit=k
    )
    return response

In [ ]:
response =retrieve_embeddings(
    query="Android 12 Tablet for Kids",
    k=5)
response.points